# Grok

**Grok** is xAI's family of frontier LLMs (Grok 4 / 3 / 3-mini), served through an **OpenAI-compatible REST API** at `api.x.ai`. Its signature feature is **Live Search** — first-class, real-time access to **X (Twitter)** and the web — plus large context, reasoning modes, vision, and tool calling.

**Domain:** Proprietary Models & Coding AI  ·  **from study list**  ·  **runnable:** yes  ·  _needs `XAI_API_KEY` (live cells gate on `os.getenv`)_

## 1. What & Why

**Grok** is the LLM line from **xAI** (Elon Musk's AI company). You meet it two ways:

1. **The chatbot** — built into X/Twitter and the standalone Grok app, with a deliberately irreverent persona.
2. **The API** — `https://api.x.ai/v1`, deliberately **OpenAI-compatible**, so you call it with the stock `openai` Python SDK by changing only `base_url` and the key. There's also a native `xai-sdk` for xAI-specific features.

**The problem it solves / why reach for it:**

- **Real-time knowledge via Live Search.** Grok can pull *current* information from **X**, the web, and news at request time — handing back cited results. For "what is happening right now" questions (breaking news, live sentiment, trending topics), it's a first-class capability, not a bolt-on RAG you build yourself.
- **Drop-in migration.** Because the API mirrors OpenAI's `chat/completions` shape, an existing OpenAI codebase switches to Grok by repointing the base URL. Minimal rewrite to A/B a new model.
- **Frontier reasoning + big context.** Grok 4 targets the top of math/coding/reasoning benchmarks and offers a large context window (256K tokens), with reasoning variants (`grok-3-mini`) that expose their thinking effort.

**When NOT to:** if you need the deepest tooling ecosystem and the most battle-tested SDKs, OpenAI/Anthropic are more mature. If you need on-prem / open weights, Grok is closed and cloud-only (older Grok-1 weights were open-sourced, but the current models are not). And if you don't care about real-time X/web data, the Live Search differentiator buys you nothing — pick on price/quality alone.

## 2. Mental Model

Think of Grok as **"an OpenAI-shaped API with a live wire into X and the web."**

```
                    your code (openai SDK, base_url=https://api.x.ai/v1)
                                         |
                                         v
        +-----------------------------  xAI API  -----------------------------+
        |                                                                     |
        |   chat/completions   ->   Grok 4 / Grok 3 / Grok 3-mini (reasoning) |
        |                                   |                                 |
        |                      optional: search_parameters                    |
        |                                   |                                 |
        |                                   v                                 |
        |        Live Search  ===>  [ X / Twitter | web | news ]  ==> citations|
        +---------------------------------------------------------------------+
                                         |
                                         v
                   answer (+ citations when search ran)
```

Three things to internalize:

1. **The envelope is OpenAI's.** Messages, roles, `tools`, streaming, `response_format` — all the shapes you already know. The mental cost of adopting Grok is near zero if you know the OpenAI API.
2. **Live Search is the differentiator.** Turn it on with `search_parameters` (mode `off`/`auto`/`on`) and Grok queries X/web/news *as part of the completion*, returning citations. This is the one thing you can't easily get from the other big APIs.
3. **Reasoning is a model choice, not a flag soup.** Pick a reasoning model (`grok-3-mini`) and optionally set `reasoning_effort`; pick `grok-4` for the heaviest tasks. You don't assemble a reasoning pipeline — you choose the right model string.

## 3. Key Concepts

- **xAI** — the company; **Grok** — its model family. The API lives at `https://api.x.ai/v1`.
- **OpenAI compatibility** — the `chat/completions` (and `responses`) endpoints match OpenAI's schema. Use the `openai` SDK with `base_url="https://api.x.ai/v1"` and `api_key=XAI_API_KEY`, or the native `xai-sdk`.
- **`XAI_API_KEY`** — your credential, created at **console.x.ai**. Passed as a bearer key (env var by convention), not Google-style ADC.
- **Model strings** — e.g. `grok-4` (flagship, 256K context, reasoning), `grok-3` (general), `grok-3-mini` (small, fast, reasoning + `reasoning_effort`), `grok-2-vision-1212` (image input). Check the docs for the current roster — names and snapshots change.
- **Live Search** — pass `search_parameters` (via `extra_body` on the OpenAI SDK) to let Grok fetch real-time data. `mode`: `off` (never), `auto` (model decides), `on` (always). Choose `sources` (`x`, `web`, `news`) and limits; responses come back with **citations**. Billed per source used.
- **`reasoning_effort`** — on reasoning-capable models (`grok-3-mini`), `low`/`high` trades latency/cost for deeper thinking. Flagship Grok 4 reasons by default.
- **Tool / function calling** — same `tools` + `tool_calls` loop as OpenAI: you describe functions, the model emits a call, you execute and feed the result back.
- **Structured outputs** — `response_format={"type": "json_schema", ...}` constrains output to a schema, same as OpenAI.
- **Vision** — image-capable models accept image URLs / base64 in the multimodal message content blocks.
- **Rate limits & pricing** — token-based pricing per model (input/output), with Live Search billed separately per source. Quotas are per-account; confirm current numbers on the pricing page.

## 4. Setup

You need an **xAI account and an API key** (`console.x.ai`). The fastest path is the stock OpenAI SDK pointed at xAI:

```bash
pip install openai      # OpenAI-compatible path (recommended for migration)
pip install xai-sdk     # optional: xAI's native SDK for xAI-specific features

export XAI_API_KEY="xai-..."   # from https://console.x.ai
```

A minimal call is just the OpenAI client with two fields changed:

```python
from openai import OpenAI
client = OpenAI(api_key=os.environ["XAI_API_KEY"], base_url="https://api.x.ai/v1")
resp = client.chat.completions.create(
    model="grok-3-mini",
    messages=[{"role": "user", "content": "Hello"}],
)
print(resp.choices[0].message.content)
```

The cells below run top-to-bottom in a fresh kernel **without** the SDK or a key — every live call is gated behind an `os.getenv("XAI_API_KEY")` check.

In [ ]:
# This notebook executes with or without the SDK and an API key.
# To run the live examples, uncomment the install and set your key first:
# %pip install openai
# export XAI_API_KEY="xai-..."   # from https://console.x.ai

import os

api_key   = os.getenv("XAI_API_KEY")
base_url  = "https://api.x.ai/v1"
default_model = "grok-3-mini"

print("XAI_API_KEY  :", "set" if api_key else "(unset — live cells will be skipped)")
print("base_url     :", base_url)
print("default model:", default_model)
print("auth model   : bearer API key (OpenAI-compatible)")

## 5. Worked Examples

### Example 1 — The OpenAI-compatible migration: what changes, what doesn't (no network)

The adoption story is "repoint the client, keep the call." Below we lay the OpenAI and Grok client configs side by side so you can see exactly the two fields that differ — and that the `chat.completions.create` call is byte-for-byte identical.

In [ ]:
# Same request, two providers. Pure Python — no network, no key.

# --- OpenAI: default base_url, OPENAI_API_KEY ---
openai_client = {
    "constructor": "OpenAI()  # base_url defaults to https://api.openai.com/v1",
    "auth": "OPENAI_API_KEY",
    "model": "gpt-4o-mini",
}

# --- Grok / xAI: same SDK, repointed base_url + key ---
grok_client = {
    "constructor": "OpenAI(base_url='https://api.x.ai/v1')",
    "auth": "XAI_API_KEY",
    "model": "grok-3-mini",
}

# The create() call itself is IDENTICAL — only the model string changes:
shared_call = "client.chat.completions.create(model=MODEL, messages=[{'role':'user','content':'Hello'}])"

print("WHAT CHANGES (client construction):")
for k in ("constructor", "auth", "model"):
    print(f"  {k:12} OpenAI : {openai_client[k]}")
    print(f"  {k:12} Grok   : {grok_client[k]}")
    print()
print("WHAT STAYS THE SAME (the actual request):")
print(" ", shared_call)

### Example 2 — Estimate token cost for a Grok call (no network)

Grok bills per token (input/output) per model, plus a per-source charge when Live Search runs. A `chars / 4` heuristic is fine for back-of-envelope budgeting; the API returns exact `usage` counts on every real call.

In [ ]:
# Rough token + cost estimate for a Grok call. Pure Python, no API.
# Prices are ILLUSTRATIVE (USD per 1M tokens) — confirm at https://docs.x.ai/docs/models
PRICING = {
    "grok-4":      {"in": 3.00, "out": 15.00},
    "grok-3":      {"in": 3.00, "out": 15.00},
    "grok-3-mini": {"in": 0.30, "out": 0.50},
}
LIVE_SEARCH_PER_SOURCE = 0.025  # USD per source used when Live Search runs

def est_tokens(text: str) -> int:
    return max(1, len(text) // 4)        # ~4 chars/token for English prose

def est_cost(model: str, in_tokens: int, out_tokens: int, sources: int = 0) -> float:
    p = PRICING[model]
    tok = in_tokens / 1e6 * p["in"] + out_tokens / 1e6 * p["out"]
    return tok + sources * LIVE_SEARCH_PER_SOURCE

prompt = "Summarize today's top AI headlines and the reaction on X."
in_tok, out_tok, sources = est_tokens(prompt), 200, 5   # Live Search touched 5 sources

print(f"prompt: {prompt!r}")
print(f"input ~{in_tok} tok, output ~{out_tok} tok, live-search sources: {sources}\n")
for model in PRICING:
    no_search = est_cost(model, in_tok, out_tok)
    with_search = est_cost(model, in_tok, out_tok, sources)
    print(f"  {model:<12} ${no_search:.6f}  (+search ${with_search:.6f})")

### Example 3 — The real chat completion, gated behind the key

The actual SDK invocation. With `XAI_API_KEY` set it returns live text; without it, the cell prints the call shape so the notebook still executes cleanly end-to-end.

In [ ]:
import os

def call_grok(prompt: str, model: str = "grok-3-mini") -> str:
    from openai import OpenAI            # pip install openai
    client = OpenAI(                     # OpenAI SDK, repointed at xAI
        api_key=os.environ["XAI_API_KEY"],
        base_url="https://api.x.ai/v1",
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Reply with a single short word."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=16,
    )
    return resp.choices[0].message.content

if os.getenv("XAI_API_KEY"):
    try:
        print("Grok says:", call_grok("Reply with exactly: pong").strip())
    except Exception as e:               # bad key / quota / network
        print("Live call failed:", type(e).__name__, e)
else:
    print("XAI_API_KEY unset — skipping the live call.")
    print("Call shape: OpenAI(api_key=..., base_url='https://api.x.ai/v1')")
    print("            .chat.completions.create(model='grok-3-mini', messages=[...])")

### Example 4 — Live Search: Grok's differentiator (call shape, gated)

Live Search is the one thing you can't get cheaply from the other big APIs. You enable it by attaching `search_parameters` through `extra_body` on the OpenAI SDK. Grok queries X / web / news as part of the completion and returns **citations**. Gated behind the key; the shape is what you want in muscle memory.

In [ ]:
import os

def grok_live_search(prompt: str, model: str = "grok-3"):
    from openai import OpenAI            # pip install openai
    client = OpenAI(api_key=os.environ["XAI_API_KEY"], base_url="https://api.x.ai/v1")
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        extra_body={                     # xAI-specific: Live Search controls
            "search_parameters": {
                "mode": "on",            # off | auto | on
                "sources": [{"type": "x"}, {"type": "web"}, {"type": "news"}],
                "max_search_results": 5,
                "return_citations": True,
            }
        },
    )
    msg = resp.choices[0].message
    citations = getattr(resp, "citations", None)   # URLs Grok consulted
    return msg.content, citations

if os.getenv("XAI_API_KEY"):
    try:
        answer, cites = grok_live_search("What are people on X saying about AI this week?")
        print("answer:", (answer or "")[:300])
        print("citations:", cites)
    except Exception as e:
        print("Live call failed:", type(e).__name__, e)
else:
    print("XAI_API_KEY unset — skipping Live Search.")
    print("Shape: create(..., extra_body={'search_parameters': {'mode':'on',")
    print("        'sources':[{'type':'x'},{'type':'web'}], 'return_citations': True}})")
    print("Note: Live Search is billed per source used — use mode='auto' to let the model decide.")

## 6. Gotchas & Pitfalls

- **"OpenAI-compatible" is not "OpenAI-identical."** The common path (messages, tools, streaming, JSON schema) works unchanged, but newer/edge OpenAI features may be absent or behave differently, and xAI extras (Live Search) live in `extra_body`. Don't assume 100% parity — test the specific features you depend on.
- **Live Search costs money per source.** With `mode="on"`, every request fetches sources and is billed *on top of* tokens. Use `mode="auto"` so the model only searches when needed, and cap `max_search_results`. A loop that always searches is a quiet way to run up a bill.
- **Forgetting `base_url` hits OpenAI, not xAI.** If you reuse an OpenAI client object and forget to set `base_url="https://api.x.ai/v1"`, your `grok-*` model string 404s (or you accidentally bill the wrong account). The base URL is the one thing you must not forget.
- **Model names drift.** Grok versions and dated snapshots come and go; a hardcoded `grok-3` may be retired. Read the model list from the docs/endpoint rather than pinning a string forever, and handle `model_not_found` gracefully.
- **Reasoning models behave differently.** `grok-3-mini` and Grok 4 may emit reasoning/thinking content and ignore or reinterpret `temperature`; `reasoning_effort` is only valid on reasoning-capable models. Don't send reasoning params to non-reasoning models.
- **Persona ≠ reliability.** Grok's chatbot personality (and its willingness to be edgy) is a product choice; for production, pin a tight `system` prompt and don't rely on default tone. The API is steerable, but the brand voice can leak if you don't constrain it.
- **Rate limits and context aren't infinite.** The 256K context is for the flagship; smaller models have smaller windows. Per-account rate limits apply — handle `429` with backoff just as you would with any provider.
- **Closed and cloud-only.** Current Grok weights are not downloadable (the open-sourced **Grok-1** is an old, much weaker model). If your requirement is self-hosting or data never leaving your network, Grok's API doesn't satisfy it.

## 7. When to Use vs Alternatives

| You need… | Reach for | Why |
|---|---|---|
| **Real-time X / web data in answers** (with citations) | **Grok + Live Search** | First-class, real-time access to X and the web as part of the completion. |
| **Drop-in A/B against an OpenAI codebase** | **Grok** (OpenAI-compatible) | Repoint `base_url` + key; the request code is unchanged. |
| **Deepest tooling / most mature ecosystem** | **OpenAI** or **Anthropic** | Largest SDK, integration, and community surface; longest track record. |
| **Strong long-context coding / agentic work** | **Claude** (see `anthropic-claude-api`) | Class-leading coding + tool-use reliability and large context. |
| **Self-hosting / open weights / data residency** | **Llama / Mistral / Qwen** | Open weights you run yourself; Grok is closed and cloud-only. |

**Honest trade-offs:**

- **vs OpenAI / Anthropic** — Grok matches the frontier on benchmarks and is trivially easy to adopt from an OpenAI codebase, but the competitors have deeper ecosystems, more mature SDKs, and longer reliability track records. Grok's clear edge is **Live Search / real-time X data**.
- **vs Perplexity** (see `perplexity`) — both blend search with generation. Perplexity is search-answer-first across the open web; Grok is a general LLM with X/web search as an option, plus reasoning, vision, and tool calling under one OpenAI-shaped API.
- **vs Gemini** (see `google-gemini`) — Gemini also offers grounding/search and huge context within Google's ecosystem; Grok's differentiator is the **X firehose** specifically and the zero-friction OpenAI compatibility.
- **vs open models** (Llama, Mistral, Qwen) — those win on cost-at-scale, control, and self-hosting; Grok wins on frontier quality + real-time data with zero ops.

**Rule of thumb:** reach for Grok when you want **real-time X/web knowledge** or a **near-zero-effort** alternative to plug into an existing OpenAI pipeline. Default to OpenAI/Anthropic for the most mature ecosystem, Claude for heavy coding, and open models when you must self-host.

## 8. Resources

- **xAI API documentation (official)** — https://docs.x.ai/docs/overview
- **The Hitchhiker's Guide to Grok (quickstart)** — https://docs.x.ai/docs/tutorial
- **OpenAI-compatibility guide** — https://docs.x.ai/docs/guides/migration
- **Live Search guide** — https://docs.x.ai/docs/guides/live-search
- **Models & pricing** — https://docs.x.ai/docs/models
- **Console (create your API key)** — https://console.x.ai
- **xAI news / model announcements** — https://x.ai/news

**Related notebooks:** `perplexity` (search-native answer engine), `google-gemini` (grounding + huge context), `anthropic-claude-api` and `deepseek` for head-to-head comparison in this domain.